In [2]:
import os
import fitz
import re
import json
from datetime import datetime
from typing import Optional, List, Callable, Any, Tuple, Dict
from abc import abstractmethod, ABC
import random
import numpy as np
import pandas as pd
import copy
import nltk
from nltk.corpus import stopwords
import pickle
import itertools

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
from utils import file_handle, grouping
from utils.tree import Node

In [4]:
extract_fol = "extracted"
equipment_info_loc = os.path.join(extract_fol, "equipment_info")

In [5]:
rule_information = file_handle.load_pickle(
    os.path.join(extract_fol, "TreeStruct.pkl")
)

asset_descriptions = file_handle.load_json(
    os.path.join(equipment_info_loc, "gre_desc.json")
)

asset_descriptions["NONE"] = {"desc":"Decription not available","detailed_desc":"NONE"}

# Observation stats 

In [6]:
total_obs = list()
unq_obs = set()
for d in rule_information["rule_set"]:
    unq_obs = unq_obs.union(set(d["display_text"]["observations"]))
    total_obs.extend(d["display_text"]["observations"])

len(total_obs), len(unq_obs)

unq_obs = list(unq_obs)

In [7]:
for i, unq in enumerate(unq_obs):
    print("{} : {}".format(i,unq))

0 : Cooling coil maybe passing water. Check air line on chilled water valve
1 : Static pressure sensors need calibration, repair or replacement
2 : System is in a manual mode
3 : Lost signal to valve, check pneumatic or electrical inputs
4 : Damper is open when it is humid outside
5 : Heating valve operation
6 : Low refrigerant levels
7 : CRAC units with the chilled water valve is closed indicates a problem with controls wires from the valve to CRAC unit could be off
8 : System is not designed well for part loads on the chiller
9 : Pressure issues
10 : Opportunity for additional savings. Reset temperature to outside conditions
11 : Electrical system problems
12 : Broken belt or loose belt on Motor
13 : Motor issue
14 : Temperature sensors may need calibration or replacement
15 : Check logic on building control system
16 : Hand or overridden on when the supply fan is off
17 : Temperature sensors
18 : Dampers not working correctly
19 : Verify pump is not needed and turn off
20 : The Heat

In [8]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

/root/projects/AssetDiagnosisIQ/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
enc = model.encode(unq_obs)

In [10]:
norm = np.linalg.norm(enc, axis=1)
norm = np.outer(norm,norm) + 1e-6

In [11]:
_thresh = 0.992
cosine_sim = (enc @ enc.T)/norm
cosine_sim = np.abs(cosine_sim)

thresh = np.quantile(cosine_sim.flatten(),_thresh)
print(thresh)
for i in range(cosine_sim.shape[0]):
    for j in range(i, cosine_sim.shape[0]):
        cosine_sim[i,j] = 0

ind = np.where(cosine_sim>thresh)

observation_map = {}

edges = []
for x,y in zip(ind[0], ind[1]):
    edges.append((x,y))
    
res = grouping.getComponents(cosine_sim.shape[0], edges)


0.756036


In [12]:
obs_mapping = {}
for r in res:
    if len(r)>1:
        major = np.argmax([len(unq_obs[x]) for x in r])
        major = unq_obs[r[major]]
        
        for r0 in r:
            obs_mapping[unq_obs[r0]] = major 
    else:
        obs_mapping[unq_obs[r[0]]] = unq_obs[r[0]]

# Rule stats 

In [13]:
rule_information = file_handle.load_json(
    os.path.join("extracted/observations.rules.json")
)

In [14]:
total_rules = list()
unq_rules = set()
for d in rule_information["rule_set"]: 
    s = (d["display_text"]["rules"].replace("-","").replace("OR", "").replace("(", "").replace(")", "")).split("\n")
    s = [x.strip() for x in s]
    unq_rules = unq_rules.union(set(s))
    total_rules.extend(s)

unq_rules = list(unq_rules)

In [15]:
len(total_rules), len(unq_rules)

(633, 363)

In [16]:
unq_rules

['Preheat Valve % > 97%',
 'Outside Air Flow >/< Outside Air Flow Setpoint * 1.10%',
 'Supply Air Temperature > 90 °F  Mixed Air Temperature > 90 °F',
 'PSI',
 'Air Flow < 100 CFM IF Reporting',
 'Natural Gas Flow > 5 AND Flue Gas Oxygen % > 3%',
 'AHU Running',
 'Conductivity > 4000 umho 0.004 S',
 'Cooling Valve % > 5%',
 'SubType NOT RAS or OAU',
 'VAV Damper > 20 %',
 'Name = B3010080 AND Flue Gas Temperature < 360 °F',
 'Power > Previous Daily Average Friday Supply Power * 0.8',
 'OAT < 80 °F',
 'Subtype NOT VAV or OAU or RAU',
 'Heating Valve % > 0%',
 'Economizer Mode AND Supply Relative Humidity % Not Reporting',
 'Power Input > 0.1 KW',
 'Return Air Temperature > Setpoint Temperature',
 'Cooling Valve % > 97%',
 'OAT < 95 °F',
 'IF Reporting',
 'Poughkeepsie & Warden: Met for 4 Hours',
 'Cooling Valve % >= 5%',
 'OAT > Setpoint Temperature  OAT > Supply Air Temperature AND',
 'Propane Flow > 5 AND Flue Gas Oxygen % > 5%',
 'Schedule = 0',
 'Reporting',
 'OAT < 90 °F',
 'Coolin

In [17]:
enc = model.encode(unq_rules)

In [18]:
norm = np.linalg.norm(enc, axis=1)
norm = np.outer(norm,norm) + 1e-6

In [19]:
_thresh = 0.997
cosine_sim = (enc @ enc.T)/norm
cosine_sim = np.abs(cosine_sim)

thresh = np.quantile(cosine_sim.flatten(),_thresh)
print(thresh)
for i in range(cosine_sim.shape[0]):
    for j in range(i, cosine_sim.shape[0]):
        cosine_sim[i,j] = 0

ind = np.where(cosine_sim>thresh)

edges = []
for x,y in zip(ind[0], ind[1]):
    edges.append((x,y))
    
res = grouping.getComponents(cosine_sim.shape[0], edges)


0.9928105


In [20]:
rules_mapping = {}
for r in res:
    if len(r)>1:
        major = np.argmax([len(unq_rules[x]) for x in r])
        major = unq_rules[r[major]]

        print("Major: \n{}".format(major))
        
        for r0 in r:
            print(unq_rules[r0])
            rules_mapping[unq_rules[r0]] = major 

        print("\n")
    else:
        rules_mapping[unq_rules[r[0]]] = unq_rules[r[0]]

Major: 
Preheat Valve % < 5% AND Heating AND Reheat Valve % Not Reporting
Preheat Valve % < 5% AND Heating AND Reheat Valve % Not Reporting
Heating AND Reheat Valve % < 5% AND Preheat Valve % Not Reporting
Heating Valve % < 5% AND Preheat AND Reheat Valve % Not Reporting
Heating AND Preheat Valve % < 5% AND Reheat Valve % Not Reporting
Reheat Valve % < 5% AND Heating AND Preheat Valve % Not Reporting
Preheat AND Reheat Valve % < 5% AND Heating Valve % Not Reporting


Major: 
Met for 2 Hours
Met for 2 Hours
Met for 2 hours


Major: 
Economizer Mode = 1
Economizer Mode =1
Economizer Mode = 1


Major: 
Low Temp Chilled Water Valve % > 5%
Low Temp Chilled Water Valve % > 5%
Low Temp Chilled Water Valve % < 5%


Major: 
35 °F < OAT < 70 °F
35 °F < OAT < 70 °F
35 °F < OAT < 80 °F


Major: 
Lighting Mode = 0 AND Occupied Mode = 1
Lighting Mode = 0 AND Occupied Mode = 1
Lighting Mode = 1 AND Occupied Mode = 0


Major: 
Met for 4 hours
Met for 4 hours
Met for 4 Hours


Major: 
Outside Air Dampe

# Rule Grouping with semantic understanding

In [21]:
from sentence_transformers import SentenceTransformer, util


model = SentenceTransformer("all-mpnet-base-v2")

In [22]:
rule_repr_dict:Dict[str,str] = {x["id"]:x["display_text"]["rules"] for x in rule_information["rule_set"]}
enc = model.encode(list(rule_repr_dict.values()))
rule_index_map = {k:v for v,k in enumerate(rule_repr_dict.keys())}
index_rule_map = {k:v for v,k in rule_index_map.items()}

norm = np.linalg.norm(enc, axis=1)
norm = np.outer(norm,norm) + 1e-6

cosine_sim = (enc @ enc.T)/norm
cosine_sim = np.abs(cosine_sim)

In [23]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(cosine_sim)

plt.savefig('RRSim_heatmap.png', dpi=600, bbox_inches='tight')

<Figure size 640x480 with 2 Axes>

In [54]:
rule_codition_similarity = np.argsort(cosine_sim, axis=-1)

rule_id_similarity_map = {}
for k,v in rule_index_map.items():
    rule_id_similarity_map[k] = [index_rule_map[x] for x in rule_codition_similarity[v]]

# Dataset Creation 

In [55]:
extract_fol = "extracted"
equipment_info_loc = os.path.join(extract_fol, "equipment_info")

random.seed(420)
np.random.seed(420)

In [56]:
rule_information = file_handle.load_pickle(
    os.path.join(extract_fol, "TreeStruct.pkl")
)

asset_descriptions = file_handle.load_json(
    os.path.join(equipment_info_loc, "gre_desc.json")
)

In [57]:
for  x in rule_information["rule_set"]:
    if x["id"] == "CT00001":
        print(x["display_text"]["rules"])

In [58]:
from dataset_utils import creation, config


dataset_config = config.DatasetCreationConfig({
    "num_options":4,
    "question_template_mode":config.QuestionTemplateMode.random,
    "question_template_random_count" : 1
})

In [59]:
dataset = creation.create_dataset(
    rule_information,
    obs_mapping,
    {k:v["desc"] for k,v in asset_descriptions.items()},
    rule_id_similarity_map,
    dataset_config
    )

In [60]:
len(dataset["questions"])

9065

In [ ]:
#creation.save_dataset(
#    dataset,
#    "simpleV5",
#    "datasets"
#)

In [105]:
from dataset_utils.reader import ADIQDataset
from dataset_utils.outputs import to_basic_prompt

ds = ADIQDataset("datasets/simpleV5")

In [106]:
len(ds.questions)

9065

In [107]:
question_template = """
## Asset Description:
{asset_type}: {asset_description}

## Conditions:
{conditions}

## How long the conditions were met:
{temporal_condition}

{question_prompt}
{options}
"""

In [108]:
print(to_basic_prompt(ds.questions[2000],question_template, ds.asset_descriptions))


## Asset Description:
AHU: Air Handling Unit: A device used to condition and circulate air as part of a heating, ventilating, and air-conditioning (HVAC) system.

## Conditions:
- AHU Running
- Enthalpy > 29 BTU/lb
- Humidification Valve % > 2%

## How long the conditions were met:
Met for 1 Hour

From the given options, which one MOST likely corresponds to the observed asset behavior?
A). Pump problems (VFD problems)
B). Check the logic in the BMS to see how it is set up
C). Static pressure sensors need calibration, repair or replacement
D). Too many CRAC units are running for the amount of load



In [109]:
ds.questions[2000].rule_id, ds.questions[2000].correct

(44, [False, True, False, False])

In [110]:
q_type_stats = {"p":0,"n":0}
for q in ds.questions:
    if q.question_type == "positive":
        q_type_stats["p"] += 1
    else:
        q_type_stats["n"] += 1

In [111]:
q_type_stats

{'p': 5240, 'n': 3825}

In [112]:
opt_comp = {}
for q in ds.questions:
    try:
        opt_comp[len(q.options)] += 1
    except KeyError as er:
        opt_comp[len(q.options)] = 1

In [113]:
opt_comp

{3: 3825, 4: 5240}